# 🚀 PrismPrice: End-to-End Multimodal Pricing Pipeline, Optuna HPO & Empirical Report Generator
### Kaggle Notebook Execution Guide (Real-Time Unbuffered Logging & Fail-First Mode)
This notebook seamlessly connects with the [A_ML_25 GitHub Repository](https://github.com/arpitkumar2004/A_ML_25.git) to execute:
1. **Real-Time Unbuffered Logging & Fail-First Mode**: Pushes stdout progress lines live to the notebook pane without delay, halting instantly on any subprocess exception.
2. **Automatic Repository Synchronization**: Clones repo and installs required dependencies (Optuna, LightGBM, XGBoost, CatBoost, PyTorch).
3. **Hardware Acceleration Diagnostic**: Auto-detects PyTorch CUDA GPU devices and sets GPU acceleration (`device='cuda'`, `tree_method='hist'`, `task_type='GPU'`).
4. **5 Domain Pricing Feature Engineering Sets**: Sub-linear quantity elasticity ($\text{Quantity}^{0.75}$), mass density ratios, unit per-pack ratios, text complexity signals, and missing image indicators.
5. **Optuna Bayesian Hyperparameter Optimization (TPE Sampler)**: Executes Bayesian HPO across LightGBM, XGBoost, CatBoost, ExtraTrees, and Ridge hyperparameters.
6. **5-Fold Cross-Validation Model Suite & Stacking**: Trains base models and fits an out-of-fold `RidgeCV` meta-learner with automated L2 regularization tuning.
7. **15 Publication Figures & 7 JSON/CSV Reports**: Auto-generates all empirical plots (`docs/`) and metrics (`experiments/reports/`).
8. **1-Click Artifact Zipping**: Archives all generated reports, plots, and test predictions into `experiments_reports_and_docs.zip` for instant download.

In [ ]:
# Step 1: Clone GitHub Repository & Set Up Working Directory
import os
import sys
import subprocess

os.environ["PYTHONUNBUFFERED"] = "1"
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except Exception:
        pass

REPO_URL = "https://github.com/arpitkumar2004/A_ML_25.git"
REPO_DIR = "A_ML_25"

if not os.path.exists(REPO_DIR):
    print(f"📥 Cloning GitHub repository from {REPO_URL}...", flush=True)
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"📂 Repository directory '{REPO_DIR}' already exists. Navigating into directory...", flush=True)
    os.chdir(REPO_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"✅ Current Working Directory: {os.getcwd()}", flush=True)

In [ ]:
# Step 2: Install Python Dependencies
!pip install -q scipy scikit-learn lightgbm xgboost catboost optuna matplotlib seaborn joblib pydantic fastapi

In [ ]:
# Step 3: Hardware Acceleration & GPU Diagnostic Inspection
import torch
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

print("==================================================", flush=True)
print("⚡ HARDWARE ACCELERATION & GPU DIAGNOSTIC", flush=True)
print("==================================================", flush=True)
cuda_available = torch.cuda.is_available()
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    cuda_ver = torch.version.cuda
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU ACTIVE: {device_name} ({device_count} device(s), {mem_gb:.2f} GB VRAM)", flush=True)
    print(f"✅ CUDA Toolkit Version: {cuda_ver}", flush=True)
    torch.backends.cudnn.benchmark = True
else:
    print("⚠️ GPU NOT DETECTED - Falling back to multi-core CPU execution.", flush=True)

print(f"✅ XGBoost Version: {xgb.__version__}", flush=True)
print(f"✅ LightGBM Version: {lgb.__version__}", flush=True)
print(f"✅ CatBoost Version: {cb.__version__}", flush=True)
print("==================================================", flush=True)

In [ ]:
# Step 4: Dataset Auto-Detection in Kaggle / Colab Environment
import glob

train_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True) + glob.glob("data/raw/train.csv") + glob.glob("train.csv")
test_candidates  = glob.glob("/kaggle/input/**/test.csv", recursive=True) + glob.glob("data/raw/test.csv") + glob.glob("test.csv")

train_path = train_candidates[0] if train_candidates else None
test_path  = test_candidates[0] if test_candidates else None

print(f"🔍 Training Dataset Path: {train_path if train_path else 'NOT FOUND (Synthetic Fallback Active)'}", flush=True)
print(f"🔍 Testing Dataset Path:  {test_path if test_path else 'NOT FOUND'}", flush=True)

In [ ]:
# Step 5: Execute End-to-End Report Generator, Optuna HPO & Benchmarks (Unbuffered & Fail-First)
cmd = [sys.executable, "-u", "main.py", "generate-report"]
if train_path:
    cmd.extend(["--data", train_path])

print(f"⚡ Running Real-Time Report Generator: {' '.join(cmd)}", flush=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.run(cmd, check=True, env=env)

In [ ]:
# Step 6: Display Optuna Hyperparameter Optimization Study Results
import json

hpo_file = "experiments/reports/hpo_optuna_results.json"
if os.path.exists(hpo_file):
    with open(hpo_file, "r") as f:
        hpo_data = json.load(f)
    print("🎯 Optuna Bayesian HPO Study Results:", flush=True)
    print(f"  - Total Trials Executed: {hpo_data.get('total_trials_executed')}", flush=True)
    print(f"  - Best Trial SMAPE (%):  {hpo_data.get('best_trial_smape'):.4f}%", flush=True)
    print(f"  - Best Hyperparameters: {json.dumps(hpo_data.get('best_hyperparameters'), indent=4)}", flush=True)
else:
    print("⚠️ HPO file not found.", flush=True)

In [ ]:
# Step 7: Fit Final Stacker Ensemble & Generate Test Predictions (Fail-First Mode)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

if train_path:
    print("🏋️ Fitting final OOF Stacker Meta-Learner on full dataset...", flush=True)
    cmd_train = [sys.executable, "-u", "main.py", "train", "--config", "configs/training/final_train.yaml", "--data", train_path, "--model", "stacker"]
    subprocess.run(cmd_train, check=True, env=env)

if test_path and os.path.exists(test_path):
    print("🔮 Generating test predictions submission.csv...", flush=True)
    cmd_predict = [sys.executable, "-u", "main.py", "predict", "--config", "configs/inference/inference.yaml", "--data", test_path, "--output", "submission.csv"]
    subprocess.run(cmd_predict, check=True, env=env)
    print("✅ Created submission.csv!", flush=True)

In [ ]:
# Step 8: Verify All 15 Generated Publication Figures & Reports
print("📊 Generated Metric Reports in experiments/reports/", flush=True)
reports = glob.glob("experiments/reports/*.json") + glob.glob("experiments/reports/*.csv")
for r in sorted(reports):
    print(f"  - {r}", flush=True)

print("\n🖼️ Generated Publication Figures in docs/", flush=True)
figures = glob.glob("docs/*.png")
for f in sorted(figures):
    print(f"  - {f}", flush=True)

In [ ]:
# Step 9: Zip All Reports & Plots for 1-Click Download
import zipfile

zip_filename = "experiments_reports_and_docs.zip"
print(f"📦 Zipping all generated reports and figures into '{zip_filename}'...", flush=True)

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in ["experiments/reports", "docs"]:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    full_p = os.path.join(root, file)
                    rel_p = os.path.relpath(full_p, os.getcwd())
                    zipf.write(full_p, rel_p)

    if os.path.exists("submission.csv"):
        zipf.write("submission.csv", "submission.csv")

print(f"🎉 DONE! Download '{zip_filename}' from your Kaggle/Colab notebook output pane.", flush=True)